# Table Generation Notebook

This notebook regenerates and displays the manuscript tables in the requested order. Edit the path variables in the first code cell if the downloaded `dataset` and `result` folders are stored outside the project root.

In [ ]:
from pathlib import Path
import html
import os
import subprocess
import sys
import time
from IPython.display import HTML, Markdown, display

FIGURES_TABLES_ROOT = Path.cwd()
if not (FIGURES_TABLES_ROOT / "table_scripts").exists():
    candidate = FIGURES_TABLES_ROOT / "figures_tables"
    if (candidate / "table_scripts").exists():
        FIGURES_TABLES_ROOT = candidate
    else:
        raise RuntimeError("Run this notebook from the figures_tables directory, or set FIGURES_TABLES_ROOT manually.")

PROJECT_ROOT = FIGURES_TABLES_ROOT.parent
RESULT_ROOT = PROJECT_ROOT / "result"
DATASET_ROOT = PROJECT_ROOT / "dataset"

TABLE_SCRIPT_DIR = FIGURES_TABLES_ROOT / "table_scripts"
TABLE_DIR = FIGURES_TABLES_ROOT / "table"
TEST_RESULT_BATCH = RESULT_ROOT / "testdataset_result_batch"
EXTERNAL_RESULT_BATCH = RESULT_ROOT / "externaldataset_result_batch"
TEST_DATASET_ROOT = DATASET_ROOT / "testdataset"
EXTERNAL_DATASET_ROOT = DATASET_ROOT / "externaldataset"

PYTHON = sys.executable


def run_command(args):
    cmd = [PYTHON] + [str(item) for item in args]
    print("$ " + subprocess.list2cmdline(cmd))
    start = time.perf_counter()
    completed = subprocess.run(
        cmd,
        cwd=FIGURES_TABLES_ROOT,
        text=True,
        capture_output=True,
        check=True,
    )
    if completed.stdout.strip():
        print(completed.stdout.strip())
    if completed.stderr.strip():
        print(completed.stderr.strip())
    print(f"Elapsed time: {time.perf_counter() - start:.2f} s")


def show_text_file(path):
    path = Path(path)
    display(Markdown(f"**{path.relative_to(FIGURES_TABLES_ROOT)}**"))
    text = path.read_text(encoding="utf-8")
    display(Markdown("```text\n" + text + "\n```"))

## 1. Table 1

Recognition ability of different endoscopic findings in four MLLMs, DLKGs, and endoscopist groups on the KGS test set.

In [ ]:
run_command([TABLE_SCRIPT_DIR / "build_overall_performance_tables.py", "--result-root", RESULT_ROOT, "--dataset-root", TEST_DATASET_ROOT, "--output-dir", TABLE_DIR])
show_text_file(TABLE_DIR / "model_group_average_report.txt")
show_text_file(TABLE_DIR / "doctor_group_average_report.txt")

## 2. Supplementary Table S1

Pairwise comparison of model classification accuracy on the KGS test set.

In [ ]:
run_command([TABLE_SCRIPT_DIR / "build_pairwise_accuracy_comparison_table.py", "--dataset", "test", "--result-root", TEST_RESULT_BATCH, "--output-dir", TABLE_DIR])
show_text_file(TABLE_DIR / "statistical_comparison_results.txt")

## 3. Supplementary Table S2

Comparison of specificity, precision, recall, and F1 Score between KGS-Gemini 3, the other three models, and DLKGs on the KGS test set.

In [ ]:
run_command([TABLE_SCRIPT_DIR / "build_pairwise_other_metrics_comparison_table.py", "--dataset", "test", "--result-root", TEST_RESULT_BATCH, "--dataset-root", TEST_DATASET_ROOT, "--output-dir", TABLE_DIR])
show_text_file(TABLE_DIR / "other_metrics_comparison_results.txt")

## 4. Supplementary Table S3

Summary of the performance of models and human groups in recognizing 18 endoscopic findings on the KGS test set.

In [ ]:
run_command([TABLE_SCRIPT_DIR / "build_subcategory_performance_table.py", "--dataset", "test", "--result-root", RESULT_ROOT, "--dataset-root", TEST_DATASET_ROOT, "--output-dir", TABLE_DIR])
show_text_file(TABLE_DIR / "subcategory_performance_metrics.txt")

## 5. Supplementary Table S10

Summary of the performance of KGS-Gemini 3 and DLKGs in recognizing 18 endoscopic findings on the KGS external validation set.

In [ ]:
run_command([TABLE_SCRIPT_DIR / "build_subcategory_performance_table.py", "--dataset", "external", "--result-root", RESULT_ROOT, "--dataset-root", EXTERNAL_DATASET_ROOT, "--output-dir", TABLE_DIR])
show_text_file(TABLE_DIR / "external_subcategory_performance_metrics.txt")

## 6. Supplementary Table S11

Pairwise comparison of KGS-Gemini 3 and DLKGs classification accuracy on the KGS external validation set.

In [ ]:
run_command([TABLE_SCRIPT_DIR / "build_pairwise_accuracy_comparison_table.py", "--dataset", "external", "--result-root", EXTERNAL_RESULT_BATCH, "--output-dir", TABLE_DIR])
show_text_file(TABLE_DIR / "external_statistical_comparison_results.txt")

## 7. Supplementary Table S12

Comparison of specificity, precision, recall, and F1 Score between KGS-Gemini 3 and DLKGs on the KGS external validation set.

In [ ]:
run_command([TABLE_SCRIPT_DIR / "build_pairwise_other_metrics_comparison_table.py", "--dataset", "external", "--result-root", EXTERNAL_RESULT_BATCH, "--dataset-root", EXTERNAL_DATASET_ROOT, "--output-dir", TABLE_DIR])
show_text_file(TABLE_DIR / "external_other_metrics_comparison_results.txt")